# Day 15: CUDA Thread、Block、Grid

i=blockIdx.x*blockDim.x+threadIdx.x；尾部 i>=N 的线程不读写。C++ 示例独立于 PyTorch ABI，需 nvcc 与适用 GPU 架构。

参考源码：`learning/kernels/vector_reduce.cu`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY15_21.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 15完整讲义](../docs/DAY15_CUDA_INDEXING.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

把grid看成整张工单，block是小组，thread是组内工人：

```text
每组256人，任务257个元素
组0：全局位置0..255
组1：全局位置256..511，但只有位置256有效
```

索引 `i=blockIdx.x*blockDim.x+threadIdx.x`。
第1组第0人处理位置256，第1组第1人算出257，必须不读不写。
多启动的255位工人不是错误，缺少尾部Mask才是越界风险。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(15, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

N=257、block=256 时需多少 blocks？第二个 block 有几个有效线程？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：257个元素为什么启动512个线程？**
答：两个完整256线程block，每个线程用i<N判断是否有有效工作。

**问：warp与block一样吗？**
答：不一样，warp是硬件执行分组，block是程序定义的协作/共享内存单位。

**问：有nvcc就代表kernel能运行吗？**
答：不能，还需设备、驱动、适用架构和实际launch验证。

**问：为什么不要直接删除尾部判断？**
答：最后block可能读取/写入越界地址，即使一次未崩溃也不正确。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY15_CUDA_INDEXING.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。